## LangChain之工具Tools

### 工具Tools
工具是代理、链或LLM可以用来与世界互动的接口。它们结合了几个要素
- 工具的名称
- 工具的描述
- 该工具输入的JSON模式
- 要调用的函数
- 是否应将工具结果直接返回给用户

LangChain通过提供统一框架集成功能的具体实现。在框架内，每个功能被封装成一个工具，具有自己的输入输出及处理方法。代理接收任务后，通过大模型推理选择适合的工具处理任务。一旦选定，LangChain将任务输入传递给该工具，工具处理输入生成输出。输出经过大模型推理，可用于其他工具的输入或作为最终结果返回给用户。

Langchain地址：https://python.langchain.com/docs/how_to/#tools

API地址：https://api.python.langchain.com/en/latest/community_api_reference.html#module-langchain_community.tools

### @tool 基本用法

In [1]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息

    参数:
        city: 城市名称，如"北京"、"上海"

    返回:
        天气信息字符串
    """
    # 你的实现
    return "晴天，温度 15°C"

**自定义工具**

In [2]:
from langchain.tools import tool

@tool
def add_number(a: int, b: int) -> int:
    """add two numbers."""
    return a + b

print(add_number.name)
print(add_number.description)
print(add_number.args)

res = add_number.run({"a": 10, "b": 20})
print(res)

add_number
add two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
30


In [9]:
import requests
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.tools import tool


# 获取天气信息的函数
@tool
def get_weather(city):
    """
    获取指定城市的天气信息

    参数:
        city: 城市名称，如"北京"、"上海"

    返回:
        天气JSON信息
    """
    apiUrl = 'http://apis.juhe.cn/simpleWeather/query'   # 接口请求URL
    apiKey = os.getenv("WEATHER_API_KEY")  # 在个人中心->我的数据,接口名称上方查看
    # print(apiKey)
    # 接口请求入参配置
    requestParams = {
        'key': apiKey,
        'city': city,
    }
    
    # 发起接口网络请求
    response = requests.get(apiUrl, params=requestParams)
    # print(response)
    # 解析响应结果
    if response.status_code == 200:
        responseResult = response.json()
        return responseResult
        # 网络请求成功。可依据业务逻辑和接口文档说明自行处理。
        # print(responseResult)  
    else:
        # 网络异常等因素，解析结果异常。可依据业务逻辑自行处理。
        print('请求异常')


# result = get_weather("长沙")
# print(result)

**工具绑定到模型**

In [10]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()


model = init_chat_model(
    model="deepseek:deepseek-chat",
    temperature=0.1,
    max_tokens=2000
)

model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("北京今天天气怎么样？")

print(response)
# 检查模型是否要求调用工具
if response.tool_calls:
    print(f"AI 决定使用工具！")
    print(f"工具调用: {response.tool_calls}")
else:
    print(f"AI 直接回答（未使用工具）")
    print(f"回复: {response.content}")

content='我来帮您查询北京今天的天气情况。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 320, 'total_tokens': 372, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 64}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '174c4596-77e2-48fa-a78c-ac540a1c88e9', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019bc2a7-3896-7540-8d67-b7d4b4503510-0' tool_calls=[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_00_mqVqKBD0JFRdYBkUjVei1hC6', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 320, 'output_tokens': 52, 'total_tokens': 372, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}
AI 决定使用工具！
工具调用: [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_00_mqVqKBD0JFRdYBkUj

### 更多Tools

Tavily Search工具：Tavily的搜索API是一个专门为人工智能代理(llm)构建的搜索引擎，可以快速提供实时、准确和真实的结果。

访问Tavily（`https://tavily.com/`）注册账号并登录，获取API 密钥

In [13]:
from langchain_community.tools.tavily_search import TavilySearchResults


tool = TavilySearchResults(max_results=1)
# print(tool.invoke("目前市场上黄金的售价是多少?"))


# 初始化大模型
model = init_chat_model(
    model="deepseek:deepseek-chat",
    temperature=0.1,
    max_tokens=2000
)

model_with_tools = model.bind_tools([tool])

response = model_with_tools.invoke("目前市场上黄金的售价是多少？")

# print(response)

from langchain_core.messages import ToolMessage
# 4. 如果模型请求调用工具
if response.tool_calls:
    tool_call = response.tool_calls[0]

    tool_result = tool.invoke(tool_call["args"])
    print(tool_result)
    tool_message = ToolMessage(
        content=str(tool_result),
        tool_call_id=tool_call["id"]
    )

    # 5. 把工具结果再喂回模型
    final_response = model.invoke([
        response,
        tool_message
    ])

    print(final_response)
else:
    print(response.content)

[{'title': '中国建设银行-市场行情', 'url': 'https://www.ccb.com/chn/home/gjs_sp/scxq/index.shtml', 'content': '上海黄金交易所贵金属行情数据由世华财讯提供，仅供参考，据此投资，风险自担。\n\n伦敦现货贵金属参考报价\n\n| 产品名称 | 最新价格 | 涨跌（%） |\n| 黄金 | 2620.85 | ↑ 24.92 |\n| 铂金 | 931.48 | ↑ 9.99 |\n| 钯金 | 917.84 | ↑ 8.86 |\n| 白银 | 29.60 | ↑ 0.02 |\n| Au9995 | 609.65 | ↓ 5.91 |\n| Au9999 | 614.53 | ↓ 1.97 |\n| AU100g | 615.18 | ↓ 3.43 |\n| AU T+D | 613.99 | ↓ 2.30 |\n| 铂金9995 | 226.00 | ↓ 2.16 |\n| 白银T+D(千克) | 7508.00 | ↓ 172.00 |\n\n更新时间：2024-12-19 单位：美元/盎司\n\n 走势图\n 日\n 周\n 月\n 黄金\n\n  + 黄金\n  + 铂金\n  + 钯金\n  + 白银\n\n相关市场指数\n\n| 产品名称 | 最新价格 | 涨跌（%） |\n| 道琼工业 | 31283.45 | ↓ 0.32 |\n| 标普500 | 3585.62 | ↓ 1.51 |\n| 恒生指数 | 26589.39 | ↓ 1.79 |\n\n更新时间：2025-11-14\n\n 走势图\n 日\n 周\n 月\n + 美元指数\n  + 道琼工业\n  + 标普500\n  + 欧元/美元\n\n账户商品行情\n\n 账户原油\n 账户铜\n 账户大豆\n\n账户原油 Brent [...] |  | 产品名称 | 涨跌 | 最新价 | 买价一 | 买量一 | 卖价一 | 卖量一 | 开盘价 | 最低价 | 最高价 | 发布时间 |\n ---  ---  ---  ---  ---  --- |\n|  | Ag(T+D) | 803 | 18765 | 18764 | 10 | 18768 | 10 | 18200 | 18187 | 18844